# Solution F1 · PCA or feature selection? Three data shapes, five strategies

**Type**: Compare

## Objective

Compare five dimensionality reduction strategies in front of a logistic regression (no reduction, `PCA(5)`, `PCA(15)`, selection of 5 features with `SelectKBest`, `PCA(5)` without a scaler) on three kinds of data: **redundant** features (5 latent factors in 40 columns), signal in
**low-variance directions** and **sparse** signal (5 informative columns out of 60). Average over 3 datasets per kind.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def generate(kind, seed=0, n=400):
    rng = np.random.default_rng(seed)
    if kind == "redundant":                        # 5 latent factors explain 40 correlated columns; y depends on 2 factors
        L = rng.normal(size=(n, 5))
        X = L @ rng.normal(size=(5, 40)) + rng.normal(0, 1, (n, 40))
        y = (L[:, 0] + L[:, 1] + rng.normal(0, 1, n) > 0).astype(int)
    elif kind == "low variance":                   # 20 high-variance noise columns and 3 low-variance informative ones
        z = rng.normal(size=(n, 3))
        y = (z.sum(1) + rng.normal(0, 0.5, n) > 0).astype(int)
        X = np.c_[rng.normal(0, 5, (n, 20)), 0.5 * z]
    else:                                          # 5 informative columns out of 60
        X = rng.normal(size=(n, 60))
        y = (X[:, :5].sum(1) + rng.normal(0, 1, n) > 0).astype(int)
    return X, y


cv = StratifiedKFold(5, shuffle=True, random_state=0)
kinds = ["redundant", "low variance", "sparse"]

## Commented solution

In [ ]:
def logistic():
    return LogisticRegression(max_iter=2000)


models = {
    "none": lambda: make_pipeline(StandardScaler(), logistic()),
    "PCA(5)": lambda: make_pipeline(StandardScaler(), PCA(5), logistic()),
    "PCA(15)": lambda: make_pipeline(StandardScaler(), PCA(15), logistic()),
    "Select(5)": lambda: make_pipeline(StandardScaler(), SelectKBest(f_classif, k=5), logistic()),
    "PCA(5) raw": lambda: make_pipeline(PCA(5), logistic()),
}

In [ ]:
table = pd.DataFrame(index=kinds, columns=list(models), dtype=float)
for kind in kinds:
    for name, create in models.items():
        table.loc[kind, name] = np.mean([cross_val_score(create(), *generate(kind, seed), cv=cv).mean() for seed in range(3)])
table.round(3)

In [ ]:
best = table.idxmax(axis=1).to_dict()
pca_loses = sorted(t for t in kinds if table.loc[t, "PCA(5)"] < table.loc[t, "none"] - 0.05)
pca_ties = sorted(t for t in kinds if abs(table.loc[t, "PCA(5)"] - table.loc[t, "none"]) < 0.03)
scaler_matters = (table["PCA(5)"] - table["PCA(5) raw"]).to_dict()
print("best:", best, "\nPCA(5) loses on:", pca_loses, "| ties on:", pca_ties, "\neffect of the scaler on PCA(5):", {k: round(v, 3) for k, v in scaler_matters.items()})

## Check

In [ ]:
with verify():
    check(list(models) == ["none", "PCA(5)", "PCA(15)", "Select(5)", "PCA(5) raw"], "five models with the given names", "none, PCA(5), PCA(15), Select(5), PCA(5) raw")
    check(table.shape == (3, 5) and table.notna().all().all(), "complete 3 x 5 table", "one accuracy for each kind and model")
    check(table.loc["low variance", "PCA(5)"] < table.loc["low variance", "none"] - 0.1 and table.loc["sparse", "PCA(5)"] < 0.6, "PCA(5) fails with low-variance signal (over 0.1 below 'none') and with sparse signal (below 0.6, almost chance)", "compare the rows of the table")
    check(pca_loses == ["low variance", "sparse"] and pca_ties == ["redundant"], "PCA(5) loses on 'low variance' and 'sparse'; it ties on 'redundant'", "difference between PCA(5) and none")
    check(best["low variance"] == "Select(5)" and best["sparse"] == "Select(5)", "supervised selection is the best when the signal is in a few columns", "idxmax per row")
    check(scaler_matters["low variance"] > 0.1 and abs(scaler_matters["redundant"]) < 0.05, "the scaler matters a lot for 'low variance' (over 0.1) and almost nothing for 'redundant'", "PCA(5) with scaler minus PCA(5) raw")
done()

## What you learned

- **Redundant** (5 factors in 40 columns): all the strategies sit between 0.79 and 0.80, `PCA(5)` (0.788) included: the 5 components hold the signal and the remaining 35 directions are noise. PCA compresses without losing anything; in accuracy it gains nothing, because a regularized logistic regression
  already handles redundancy well (differences of one point are noise).
- **Low variance:** `PCA(5)` collapses (0.64 against 0.88): the 5 highest-variance directions are the noise columns. With 15 components it recovers part of the signal (0.76), and without a scaler it is at chance (0.51). It is the scenario of C1 on a classification problem.
- **Sparse** (5 columns out of 60): `PCA(5)` is almost at chance (0.55): all the columns have the same variance and PCA has no reason to prefer the informative ones. **Supervised selection** (`SelectKBest`) wins when the signal is in a few columns, and on this data it even beats no reduction
  (0.87 against 0.83): it removes noise by using `y`.
- **Rule of choice:** PCA when you want to compress redundant columns or visualize, and you know that variance and information coincide; selection (or L1 regularization) when you think that a few columns matter; no reduction with small data and regularized models. Always compare in cross-validation with the "no reduction" baseline.

## Common pitfall

Putting PCA in every pipeline "to reduce noise". PCA reduces noise only if the noise has less variance than the signal; with low-variance or sparse signal it amplifies the noise relatively. It has to earn its place against the baseline, like every step of a pipeline.

## Interview question

> *Click on a question to show its answer.*

<details><summary>PCA or feature selection: how do you choose?</summary>

It depends on where I think the signal is. If the columns are many and redundant (correlated sensors, pixels) PCA compresses well without losing anything. If a few predictors matter and the rest is noise, PCA mixes them with the noise and supervised selection (filters, L1, trees) works better. If the
signal is in low-variance directions, PCA throws it away. In practice I try both in a pipeline with cross-validation and compare them with the no-reduction baseline.

</details>